# AC13 - Extração de Dados: Climatempo

1. Função `get_weather(id_locale)` que consulta a API do Climatempo e retorna o clima atual da cidade correspondente.
2. DataFrame com o clima de todas as cidades **do Brasil** listadas no sitemap `previsoes-cidades-agora.xml`.

In [ ]:
import re
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": "https://www.climatempo.com.br/",
    "Origin": "https://www.climatempo.com.br"
}

def get_weather(id_locale, session=None):
    url = f"https://www.climatempo.com.br/json/myclimatempo/user/weatherNow?idlocale={id_locale}"
    req = session or requests
    response = req.post(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    dados = response.json()
    return dados["data"]["getWeatherNow"][0]["data"][0]

In [ ]:
# Teste: São Paulo (idlocale=3477)
clima = get_weather(3477)
clima

In [ ]:
# Exibindo as principais informações
weather = clima["weather"]
locale = clima["locale"]

print(f"Cidade: {locale['city']} - {locale['uf']}")
print(f"Data: {weather['date']} ({weather['dayWeek']})")
print(f"Temperatura: {weather['temperature']}°C (sensação {weather['sensation']}°C)")
print(f"Condição: {weather['condition']}")
print(f"Umidade: {weather['humidity']}%")
print(f"Vento: {weather['windVelocity']} km/h {weather['windDirection']}")
print(f"Pressão: {weather['pressure']} hPa")

## 2. DataFrame com o clima de 50 cidades do Brasil

O sitemap `previsoes-cidades-agora.xml` lista 8.470 cidades, incluindo estrangeiras (México, Japão, Argentina...). O sufixo do slug é ambíguo (`-ro` pode ser Romênia ou Rondônia), então o filtro usa o que a própria API retorna:

- `locale["country"] == "Brasil"`
- latitude/longitude dentro do bounding box do Brasil (lat entre -34 e 6, lon entre -74 e -28) — confirmação extra, já que há cidades vizinhas (ex.: Argentina) com coordenadas próximas

Alguns ids retornam erro 500 da própria API (ex.: `idlocale=1`) — esses são pulados. O loop percorre o sitemap até juntar **50 cidades brasileiras**.

In [ ]:
# Baixa o sitemap e extrai os idlocale das URLs
# (formato: .../previsao-do-tempo/agora/cidade/<idlocale>/<slug>)
url_sitemap = "https://www.climatempo.com.br/sitemap/previsoes-cidades-agora.xml"
xml = requests.get(url_sitemap, headers=HEADERS, timeout=30).text

ids = re.findall(r"cidade/(\d+)/", xml)
ids = list(dict.fromkeys(ids))  # remove duplicados mantendo a ordem

print(f"Cidades no sitemap: {len(ids)}")

In [ ]:
# Percorre o sitemap chamando get_weather até juntar 50 cidades do Brasil
session = requests.Session()

linhas = []
erros = 0
fora_do_brasil = 0

for id_locale in ids:
    if len(linhas) == 50:
        break
    try:
        resultado = get_weather(id_locale, session)
    except Exception:
        erros += 1  # ids quebrados na API (erro 500) ou timeout
        continue

    loc = resultado["locale"]
    w = resultado["weather"]
    lat, lon = loc.get("latitude"), loc.get("longitude")

    # filtro: só cidades do Brasil
    if loc.get("country") != "Brasil" or lat is None or lon is None \
       or not (-34 <= lat <= 6) or not (-74 <= lon <= -28):
        fora_do_brasil += 1
        continue

    # cada linha = locale + weather achatados em um dict só
    linhas.append({**{f"locale_{k}": v for k, v in loc.items()},
                   **{f"weather_{k}": v for k, v in w.items()}})

print(f"Brasil: {len(linhas)} | fora do Brasil: {fora_do_brasil} | erros: {erros}")

In [ ]:
df = pd.DataFrame(linhas)
df

In [ ]:
# Quantidade de cidades por estado
df["locale_uf"].value_counts()

In [ ]:
# Salva o resultado em CSV
df.to_csv("clima-cidades-brasil.csv", index=False)
print(f"{len(df)} cidades salvas em clima-cidades-brasil.csv")